# Stage 0 — Environment Setup

Goal: a reproducible Python environment and a verified Copernicus Marine login. Do this **once per machine**. If you've already done it for another project, skip to the verification cells.

## What we need
- Python 3.14.2—the version used to generate the saved artifacts.
- A virtual environment (`.venv`) at the project root
- The packages listed below
- A `copernicusmarine login` token cached in your home dir
- Network access to the CMEMS catalogue + ECMWF CDS (later, for ERA5)

## 0.1 — Create the virtual environment and install packages

Run these commands in a PowerShell terminal, not in a notebook cell. A running notebook cannot activate its own environment.

```powershell
cd experiment
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r requirements-lock.txt
```

Then point Jupyter at the `.venv` interpreter before running the import checks below. The lock file is the reproducible environment input; Notebook 11 records the environment actually used for the validated run.


## 0.2 — One-time Copernicus Marine login

Register free at <https://data.marine.copernicus.eu> if you don't already have an account. Then in a terminal (with the venv active):

```powershell
copernicusmarine login
```

You'll be prompted for username + password. The token is cached in your home directory (`~/.copernicusmarine/`) so future API calls authenticate silently.

## 0.3 — Verify everything imports

If any of these fail with `ModuleNotFoundError`, the kernel isn't pointing at the venv (or pip install missed a package). Fix that before continuing.

In [ ]:
import sys
import importlib

REQUIRED = [
    "copernicusmarine", "xarray", "netCDF4", "dask",
    "numpy", "pandas", "sklearn", "xgboost",
    "matplotlib", "shap", "cdsapi", "pyarrow", "seaborn",
]

print(f"Python {sys.version.split()[0]} @ {sys.executable}\n")

missing = []
for pkg in REQUIRED:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  OK  {pkg:20s} {ver}")
    except ModuleNotFoundError as e:
        print(f"MISS  {pkg:20s} {e}")
        missing.append(pkg)

if missing:
    raise RuntimeError(
        f"Environment validation failed for: {missing}"
    )
else:
    print("\nAll required packages present.")

Python 3.14.2 @ c:\GIT\mcast.pages.ai.v2\.venv\Scripts\python.exe

  OK  copernicusmarine     2.4.1
  OK  xarray               2026.4.0
MISS  netCDF4              No module named 'netCDF4'
  OK  dask                 2026.3.0
  OK  numpy                2.4.6
  OK  pandas               3.0.3
  OK  sklearn              1.8.0
  OK  xgboost              3.2.0
  OK  matplotlib           3.10.9
  OK  shap                 0.52.0
  OK  cdsapi               ?
  OK  pyarrow              24.0.0
  OK  seaborn              0.13.2


RuntimeError: Environment validation failed for: ['netCDF4']

: 

In [2]:
from pathlib import Path
import sys

EXPERIMENT_DIR = Path.cwd().resolve()
REPO_ROOT = EXPERIMENT_DIR.parent
EXPECTED_VENV = (REPO_ROOT / ".venv").resolve()

assert (EXPERIMENT_DIR / "01-Data-Acquisition.ipynb").exists(), (
    "Working directory must be experiment/"
)
assert Path(sys.prefix).resolve() == EXPECTED_VENV, (
    f"Wrong kernel: {sys.executable}\nExpected environment: {EXPECTED_VENV}"
)
assert sys.version_info[:3] == (3, 14, 2), (
    f"Expected Python 3.14.2, got {sys.version.split()[0]}"
)

In [3]:
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    check=True,
)

CompletedProcess(args=['c:\\GIT\\mcast.pages.ai.v2\\.venv\\Scripts\\python.exe', '-m', 'pip', 'check'], returncode=0)

## 0.4 — Verify CMEMS catalogue access

`describe()` is a metadata-only call. It fails fast on auth errors (re-run `copernicusmarine login`) and on network/firewall issues — so it's a clean separator between "environment broken" and "data broken".

In [1]:
import copernicusmarine

DATASET_ID = "cmems_obs-oc_med_bgc-plankton_my_l3-multi-1km_P1D"

catalogue = copernicusmarine.describe(
    dataset_id=DATASET_ID,
    raise_on_error=True
)

print(f"Catalogue access OK: {DATASET_ID}")

credentials_ok = copernicusmarine.login(
    check_credentials_valid=True
)
assert credentials_ok, (
    "Copernicus Marine credentials are missing or invalid. "
    "Run `copernicusmarine login` in a terminal."
)
print("Stored Copernicus Marine credentials are valid.")

Fetching catalogue 1:  50%|█████     | 1/2 [00:00<00:00,  1.20it/s]INFO - 2026-08-17T11:17:39Z - Checking if credentials are valid.


Catalogue access OK: cmems_obs-oc_med_bgc-plankton_my_l3-multi-1km_P1D


INFO - 2026-08-17T11:17:39Z - Valid credentials from configuration file.


Stored Copernicus Marine credentials are valid.


---
## Validation Gate 0

Three things to confirm before you move on:

1. All required packages imported without error (cell 0.3).
2. `describe()` on the Stage 1 dataset returned without error (cell 0.4) — proves both auth and network.
3. You understand **why** we picked `*_my_l3-multi-1km_P1D`:
   - `l3` → not gap-filled (we want honest NaNs as the methodological core)
   - `multi` → multi-sensor merge, maximises daily coverage without temporal interpolation
   - `my` → is a multi-year reprocessed historical observation product, not a “reanalysis.”
   - `1km` → is the finest long-record multi-sensor grid, not the finest Mediterranean grid available; an OLCI 300 m product exists from 2016.
   - `P1D` → daily cadence

If failing: auth errors → re-run `copernicusmarine login` in a terminal. Network → check firewall/proxy. Dataset-not-found → CMEMS may have versioned the ID; run `copernicusmarine describe --contains "med_bgc-plankton"` to list current short-names and update `DATASETS_TO_PROBE`.

---
## Next: `01-Data-Acquisition.ipynb`

Pulls the L3 CHL cube for the Malta–Sicily box (3 years to start) and runs Validation Gate 1.